# 🧠 Neural Networks & NLP — A Beginner's Notebook

Welcome! This notebook is your step-by-step guide to understanding:
- **Neural Networks**: How machines "learn" from data
- **NLP (Natural Language Processing)**: How machines understand and process human language

Every cell is explained in plain English before you run it. Take your time!

---
**Prerequisites**: Basic Python knowledge (variables, loops, functions)

**How to use this notebook**: Read the explanation 📖, then run the code cell ▶️, then look at the output 👀

---
# PART 1: The Building Block — A Single Neuron

Before we build a neural *network*, we need to understand one single neuron.

## What is a Neuron?
Think of a neuron like a **decision-maker**:
- It receives some **inputs** (numbers)
- It multiplies each input by a **weight** (how important is this input?)
- It adds a **bias** (a baseline nudge)
- It produces one **output**

**Real-world analogy**: Deciding whether to bring an umbrella 🌂
- Input 1: Is it cloudy? (0 = no, 1 = yes) → weight = 0.8 (very important)
- Input 2: Is it summer? (0 = no, 1 = yes) → weight = -0.3 (less important, summer showers are brief)
- Bias: -0.5 (you prefer NOT carrying an umbrella by default)

The neuron adds it all up and if the result is > 0, bring the umbrella!

In [ ]:
# ============================================================
# CELL 1: A single neuron from scratch (no libraries!)
# ============================================================
# We'll manually compute what one neuron does.
# Formula: output = (input1 * weight1) + (input2 * weight2) + bias

# --- Inputs (our features) ---
is_cloudy  = 1   # Yes, it's cloudy
is_summer  = 0   # No, it's not summer

# --- Weights (learned importance of each input) ---
weight_cloudy = 0.8
weight_summer = -0.3

# --- Bias (baseline shift) ---
bias = -0.5

# --- Compute the neuron output ---
output = (is_cloudy * weight_cloudy) + (is_summer * weight_summer) + bias

print(f"Neuron output: {output}")

# Make a decision based on the output
if output > 0:
    print("Decision: Bring the umbrella! 🌂")
else:
    print("Decision: Leave the umbrella at home ☀️")

# Try changing is_cloudy to 0 and see what happens!

## What is an Activation Function?

The raw neuron output is just a number. An **activation function** squishes or shapes that number into a useful range.

The most popular one is **ReLU** (Rectified Linear Unit):
- If output < 0 → return 0 ("be quiet")
- If output > 0 → return the output as-is ("speak up")

Another one is **Sigmoid**:
- Squishes ANY number into the range (0, 1)
- Useful when you want a **probability** as output

Why do we need these? Without activation functions, no matter how many layers we stack, the network can only learn **straight lines**. Activation functions let it learn **curves and complex patterns**.

In [ ]:
# ============================================================
# CELL 2: Activation Functions visualized
# ============================================================
# We'll plot ReLU and Sigmoid side-by-side so you can SEE them.

import numpy as np
import matplotlib.pyplot as plt

# Create a range of numbers from -5 to +5
x = np.linspace(-5, 5, 200)

# --- Define the two activation functions ---
def relu(x):
    return np.maximum(0, x)   # Keep positives, zero out negatives

def sigmoid(x):
    return 1 / (1 + np.exp(-x))  # Squish everything to (0, 1)

# --- Plot them ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ReLU plot
axes[0].plot(x, relu(x), color='steelblue', linewidth=2.5)
axes[0].axhline(0, color='gray', linewidth=0.8, linestyle='--')
axes[0].axvline(0, color='gray', linewidth=0.8, linestyle='--')
axes[0].set_title('ReLU Activation', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Input')
axes[0].set_ylabel('Output')
axes[0].annotate('Negative inputs → 0', xy=(-3, 0.2), fontsize=10, color='red')
axes[0].annotate('Positive inputs pass through', xy=(0.5, 3), fontsize=10, color='green')

# Sigmoid plot
axes[1].plot(x, sigmoid(x), color='darkorange', linewidth=2.5)
axes[1].axhline(0.5, color='gray', linewidth=0.8, linestyle='--')
axes[1].set_title('Sigmoid Activation', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Input')
axes[1].set_ylabel('Output (probability)')
axes[1].annotate('Always between 0 and 1', xy=(-4.5, 0.85), fontsize=10, color='darkorange')

plt.tight_layout()
plt.suptitle('Activation Functions', fontsize=16, y=1.02)
plt.show()

# Test the functions on our neuron output from Cell 1
neuron_output = 0.3
print(f"Raw neuron output: {neuron_output}")
print(f"After ReLU:       {relu(neuron_output):.4f}")
print(f"After Sigmoid:    {sigmoid(neuron_output):.4f}  ← interpreted as {sigmoid(neuron_output)*100:.1f}% probability")

---
# PART 2: Building a Neural Network with NumPy

Now let's stack neurons into **layers** to build a real network — without any ML library!

## Our Task: XOR Problem
XOR (exclusive OR) is a classic problem that a single neuron **cannot** solve:

| Input A | Input B | XOR Output |
|---------|---------|------------|
| 0       | 0       | 0          |
| 0       | 1       | 1          |
| 1       | 0       | 1          |
| 1       | 1       | 0          |

This requires a **hidden layer** — two layers of neurons. This is why deep networks are powerful!

In [ ]:
# ============================================================
# CELL 3: Neural Network from scratch — learning XOR
# ============================================================
# We build a 2-layer network and train it using backpropagation.
# Don't worry about every detail — focus on the big picture!

import numpy as np
np.random.seed(42)  # For reproducibility

# --- XOR Dataset ---
X = np.array([[0,0], [0,1], [1,0], [1,1]])  # Inputs
y = np.array([[0],   [1],   [1],   [0]])    # Correct outputs

# --- Network Architecture ---
# Input layer:  2 neurons (A, B)
# Hidden layer: 4 neurons
# Output layer: 1 neuron  (prediction)

# Initialize weights randomly (small numbers)
W1 = np.random.randn(2, 4) * 0.5   # 2 inputs → 4 hidden neurons
b1 = np.zeros((1, 4))               # Biases for hidden layer
W2 = np.random.randn(4, 1) * 0.5   # 4 hidden → 1 output
b2 = np.zeros((1, 1))               # Bias for output

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_deriv(x):
    return x * (1 - x)  # Derivative of sigmoid (needed for learning)

# --- Training Loop ---
learning_rate = 0.5
loss_history = []

for epoch in range(5000):
    # === FORWARD PASS: compute predictions ===
    hidden = sigmoid(X @ W1 + b1)          # Hidden layer
    output = sigmoid(hidden @ W2 + b2)     # Output layer

    # === LOSS: how wrong are we? ===
    loss = np.mean((y - output) ** 2)      # Mean Squared Error
    loss_history.append(loss)

    # === BACKWARD PASS: who's to blame? adjust weights ===
    d_output = (y - output) * sigmoid_deriv(output)
    d_hidden = (d_output @ W2.T) * sigmoid_deriv(hidden)

    W2 += hidden.T @ d_output * learning_rate
    b2 += np.sum(d_output, axis=0) * learning_rate
    W1 += X.T @ d_hidden * learning_rate
    b1 += np.sum(d_hidden, axis=0) * learning_rate

# --- Results ---
print("Training complete!\n")
print("Input A | Input B | Expected | Predicted | Correct?")
print("-" * 55)
for i in range(4):
    predicted = output[i][0]
    expected = y[i][0]
    correct = "✅" if abs(predicted - expected) < 0.1 else "❌"
    print(f"  {int(X[i][0])}     |    {int(X[i][1])}    |    {expected}     |   {predicted:.3f}   |  {correct}")

# --- Plot the learning curve ---
plt.figure(figsize=(8, 4))
plt.plot(loss_history, color='crimson', linewidth=1.5)
plt.title('Learning Curve — Loss Goes Down as Network Learns', fontsize=13)
plt.xlabel('Epoch (training step)')
plt.ylabel('Loss (lower = better)')
plt.annotate('High error at start', xy=(100, loss_history[100]),
             xytext=(500, 0.22), arrowprops=dict(arrowstyle='->', color='gray'), fontsize=9)
plt.annotate('Low error after learning', xy=(4500, loss_history[4500]),
             xytext=(3000, 0.18), arrowprops=dict(arrowstyle='->', color='gray'), fontsize=9)
plt.tight_layout()
plt.show()

---
# PART 3: Neural Network with PyTorch (Real ML Library)

Now we use **PyTorch** — the most popular deep learning library. Same idea, but much cleaner code and more powerful.

## Our Task: Classify Flowers 🌸
We'll use the famous **Iris dataset**:
- 150 flowers, 3 species (Setosa, Versicolor, Virginica)
- 4 measurements per flower (sepal length/width, petal length/width)
- Goal: Given measurements → predict the species

This is a **classification problem** — one of the most common in ML!

In [ ]:
# ============================================================
# CELL 4: Install and import libraries
# ============================================================
# Run this cell first. It may take a moment on first run.

# Uncomment the line below if torch is not installed:
# !pip install torch scikit-learn --quiet

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np

print(f"PyTorch version: {torch.__version__}")
print("All libraries loaded! ✅")

In [ ]:
# ============================================================
# CELL 5: Load and explore the Iris dataset
# ============================================================
# Before training, always LOOK at your data!

iris = load_iris()
X = iris.data    # Shape: (150, 4) — 150 samples, 4 features
y = iris.target  # Shape: (150,)   — 0, 1, or 2 (species index)

print("=== Dataset Overview ===")
print(f"Number of samples:  {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"Feature names: {iris.feature_names}")
print(f"Classes: {iris.target_names.tolist()}")
print(f"\nFirst 5 rows of data:")
print(X[:5])
print(f"\nCorresponding labels: {y[:5]} → {[iris.target_names[i] for i in y[:5]]}")

# Quick visualisation of two features
colors = ['#E63946', '#2A9D8F', '#F4A261']
plt.figure(figsize=(8, 5))
for i, name in enumerate(iris.target_names):
    mask = y == i
    plt.scatter(X[mask, 2], X[mask, 3], c=colors[i], label=name, s=60, alpha=0.8)
plt.xlabel('Petal Length (cm)', fontsize=12)
plt.ylabel('Petal Width (cm)', fontsize=12)
plt.title('Iris Dataset — Petal Features by Species', fontsize=13)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Notice how the species form clear clusters — our network should learn these!

In [ ]:
# ============================================================
# CELL 6: Build and train the neural network
# ============================================================
# We define the network architecture as a Python class.
# Then we train it on 80% of data and test on the other 20%.

# --- Step 1: Prepare data ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)  # 80% train, 20% test

scaler = StandardScaler()  # Normalize features to mean=0, std=1
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Convert to PyTorch tensors (PyTorch's version of NumPy arrays)
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.LongTensor(y_train)
X_test_t  = torch.FloatTensor(X_test)
y_test_t  = torch.LongTensor(y_test)

# --- Step 2: Define network architecture ---
class IrisNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Layer 1: 4 inputs → 16 hidden neurons
        self.layer1 = nn.Linear(4, 16)
        # Layer 2: 16 → 8 hidden neurons
        self.layer2 = nn.Linear(16, 8)
        # Output: 8 → 3 classes (one score per species)
        self.output = nn.Linear(8, 3)
        self.relu   = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.layer1(x))   # Layer 1 + activation
        x = self.relu(self.layer2(x))   # Layer 2 + activation
        x = self.output(x)              # Output (no activation — CrossEntropy handles it)
        return x

model    = IrisNet()
loss_fn  = nn.CrossEntropyLoss()       # Good for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.01)  # Smarter than plain gradient descent

# --- Step 3: Training loop ---
losses, accuracies = [], []

for epoch in range(200):
    model.train()
    optimizer.zero_grad()           # Clear old gradients
    predictions = model(X_train_t)  # Forward pass
    loss = loss_fn(predictions, y_train_t)  # Compute loss
    loss.backward()                 # Backpropagation
    optimizer.step()                # Update weights

    # Track accuracy every 10 epochs
    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            test_preds = model(X_test_t).argmax(dim=1)
            acc = (test_preds == y_test_t).float().mean().item()
        losses.append(loss.item())
        accuracies.append(acc * 100)

# --- Step 4: Final evaluation ---
model.eval()
with torch.no_grad():
    final_preds = model(X_test_t).argmax(dim=1)
    final_acc = (final_preds == y_test_t).float().mean().item()

print(f"Final Test Accuracy: {final_acc*100:.1f}%")

# Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs_tracked = list(range(0, 200, 10))

ax1.plot(epochs_tracked, losses, 'crimson', linewidth=2)
ax1.set_title('Loss During Training', fontsize=13)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')

ax2.plot(epochs_tracked, accuracies, 'teal', linewidth=2)
ax2.set_title('Test Accuracy During Training', fontsize=13)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.axhline(100, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

---
# PART 4: Natural Language Processing (NLP)

Computers understand numbers, not words. NLP is the process of **converting text into numbers** so a neural network can process it.

## The Journey of a Sentence:
```
"I love Python"  
  → Tokenize  → ["I", "love", "Python"]
  → Vocabulary → [42, 17, 89]
  → Embed      → [[0.2, -0.1, ...], [0.8, 0.3, ...], [0.1, 0.9, ...]]
  → Neural Net → prediction
```

We'll start with the simplest NLP task: **Sentiment Analysis** — "Is this review positive or negative?"

In [ ]:
# ============================================================
# CELL 7: Text Preprocessing — Turning words into numbers
# ============================================================
# Before any ML, we need to clean and encode our text.
# Steps: lowercase → remove punctuation → tokenize → encode

import re
from collections import Counter

# --- Sample dataset (positive=1, negative=0) ---
texts = [
    "I love this movie it is amazing",
    "This film is great and wonderful",
    "Fantastic experience highly recommend",
    "The acting was brilliant and moving",
    "I enjoyed every moment of this",
    "Terrible movie waste of time",
    "I hated this film so boring",
    "Awful experience never again",
    "Worst movie I have ever seen",
    "Disappointing and dull nothing to like",
]
labels = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]  # 1=positive, 0=negative

# --- Step 1: Clean and tokenize ---
def tokenize(text):
    text = text.lower()                      # Lowercase everything
    text = re.sub(r'[^a-z\s]', '', text)     # Remove punctuation/numbers
    return text.split()                      # Split into word list

tokenized = [tokenize(t) for t in texts]
print("Original:  ", texts[0])
print("Tokenized: ", tokenized[0])

# --- Step 2: Build vocabulary (word → index) ---
all_words = [word for sentence in tokenized for word in sentence]
word_counts = Counter(all_words)
vocab = {word: idx+2 for idx, (word, _) in enumerate(word_counts.most_common())}
vocab['<PAD>'] = 0  # Padding token (fill short sentences)
vocab['<UNK>'] = 1  # Unknown token (words not in vocabulary)

print(f"\nVocabulary size: {len(vocab)} words")
print("Sample vocab entries:")
for word in ['love', 'terrible', 'movie', 'amazing']:
    print(f"  '{word}' → index {vocab.get(word, 1)}")

# --- Step 3: Encode sentences as sequences of indices ---
MAX_LEN = 8  # Pad/truncate all sentences to this length

def encode(tokens, vocab, max_len):
    encoded = [vocab.get(t, 1) for t in tokens]  # 1 = UNK if word not found
    # Pad with zeros if too short, truncate if too long
    encoded = encoded[:max_len] + [0] * max(0, max_len - len(encoded))
    return encoded

encoded_texts = [encode(t, vocab, MAX_LEN) for t in tokenized]

print("\nEncoded text examples:")
for i in range(3):
    print(f"  '{texts[i][:40]}'")
    print(f"  → {encoded_texts[i]}")

In [ ]:
# ============================================================
# CELL 8: Word Embeddings — giving words meaning in space
# ============================================================
# An embedding turns a word index into a VECTOR (list of floats).
# Similar words end up close together in vector space.
#
# Example: 'king' - 'man' + 'woman' ≈ 'queen'
#
# We'll visualize embeddings AFTER training our sentiment model
# to see if the network learns that 'love' and 'amazing' are similar!

import torch
import torch.nn as nn

VOCAB_SIZE = len(vocab)   # How many unique words
EMBED_DIM  = 8            # Each word → 8 numbers (small for demo)

# An embedding is just a lookup table: index → vector
embedding = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=0)

# Demo: embed the first sentence
sample = torch.LongTensor([encoded_texts[0]])  # Shape: (1, 8)
embedded = embedding(sample)                   # Shape: (1, 8, 8) → 8 words × 8 dimensions

print("=== Embedding Demo ===")
print(f"Input (indices): {encoded_texts[0]}")
print(f"After embedding shape: {embedded.shape}")
print(f"  = (batch_size=1, sequence_length=8, embed_dim=8)")
print(f"\nFirst word '{tokenized[0][0]}' embedded as:")
print(f"  {embedded[0, 0].detach().numpy().round(3)}")
print("\n💡 Key idea: these numbers are LEARNED during training!")
print("   After training, 'love' and 'amazing' will have similar vectors.")

In [ ]:
# ============================================================
# CELL 9: Sentiment Classifier — full NLP model
# ============================================================
# Architecture:
#   Text → Embedding → Average pooling → Fully connected → Sigmoid → 0 or 1
#
# 'Average pooling' just averages all word vectors into one sentence vector.
# Simple but surprisingly effective!

import torch
import torch.nn as nn
import torch.optim as optim

# Prepare tensors
X_nlp = torch.LongTensor(encoded_texts)   # Shape: (10, 8)
y_nlp = torch.FloatTensor(labels)         # Shape: (10,)

# --- Define the model ---
class SentimentNet(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)
        self.relu    = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        self.dropout = nn.Dropout(0.3)  # Randomly drop 30% of neurons during training
                                        # This prevents memorization (overfitting)!

    def forward(self, x):
        embedded = self.embedding(x)         # (batch, seq, embed)
        # Average pooling: mean across all word vectors
        pooled   = embedded.mean(dim=1)      # (batch, embed)
        hidden   = self.relu(self.fc1(pooled))
        hidden   = self.dropout(hidden)
        out      = self.sigmoid(self.fc2(hidden))  # → probability (0 to 1)
        return out.squeeze()

model_nlp = SentimentNet(vocab_size=VOCAB_SIZE, embed_dim=8, hidden_dim=16)
optimizer = optim.Adam(model_nlp.parameters(), lr=0.01)
loss_fn   = nn.BCELoss()  # Binary Cross Entropy — good for yes/no predictions

# --- Train ---
losses = []
for epoch in range(300):
    model_nlp.train()
    optimizer.zero_grad()
    preds = model_nlp(X_nlp)
    loss  = loss_fn(preds, y_nlp)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

# --- Evaluate ---
model_nlp.eval()
with torch.no_grad():
    probs = model_nlp(X_nlp).numpy()

print("Results on training data:")
print(f"{'Text':<45} {'Score':>6}  {'Pred':>6}  {'True':>5}  {'OK?':>4}")
print("-" * 75)
for i, text in enumerate(texts):
    pred = 1 if probs[i] > 0.5 else 0
    mark = "✅" if pred == labels[i] else "❌"
    sentiment = "POS" if pred == 1 else "NEG"
    true_s = "POS" if labels[i] == 1 else "NEG"
    print(f"{text[:44]:<45} {probs[i]:>6.3f}  {sentiment:>6}  {true_s:>5}  {mark:>4}")

plt.figure(figsize=(8, 3))
plt.plot(losses, color='purple', linewidth=1.5)
plt.title('NLP Model Training Loss', fontsize=13)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.tight_layout()
plt.show()

---
# PART 5: Bag of Words — The Classic NLP Approach

Before deep learning, the most popular NLP technique was **Bag of Words (BoW)**:
- Create a vector of size = vocabulary
- Each position = count of that word in the sentence
- Feed the vector to a simple model

It ignores word order but works surprisingly well for many tasks!

```
Vocabulary: ["love", "hate", "movie", "great", "boring"]
"I love this great movie" → [1, 0, 1, 1, 0]  ← one count per word
"boring movie I hate"    → [0, 1, 1, 0, 1]
```

In [ ]:
# ============================================================
# CELL 10: Bag of Words + Logistic Regression (sklearn)
# ============================================================
# scikit-learn makes this very easy. Great for baselines!

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import numpy as np

# Same dataset
texts = [
    "I love this movie it is amazing", "This film is great and wonderful",
    "Fantastic experience highly recommend", "The acting was brilliant and moving",
    "I enjoyed every moment of this", "Terrible movie waste of time",
    "I hated this film so boring", "Awful experience never again",
    "Worst movie I have ever seen", "Disappointing and dull nothing to like",
]
labels = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]

# --- Bag of Words ---
bow = CountVectorizer()
X_bow = bow.fit_transform(texts).toarray()

print("=== Bag of Words ===")
print(f"Vocabulary size: {len(bow.vocabulary_)} words")
print(f"Feature matrix shape: {X_bow.shape}")
print(f"\nFirst sentence feature vector (non-zero entries):")
feature_names = bow.get_feature_names_out()
non_zero = [(feature_names[i], X_bow[0, i]) for i in np.where(X_bow[0] > 0)[0]]
print(non_zero)

# --- TF-IDF (smarter than raw counts) ---
# TF-IDF penalizes common words (like 'the', 'is') and boosts rare but meaningful words
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(texts).toarray()
print(f"\n=== TF-IDF (better than raw counts) ===")
print("TF-IDF for 'movie' (appears in many texts → lower weight):")
movie_idx = list(tfidf.vocabulary_.keys())[list(tfidf.vocabulary_.values()).index(
    tfidf.vocabulary_.get('movie', 0))]

# --- Train a Logistic Regression on BoW features ---
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_bow, labels)

print("\n=== Logistic Regression on Bag of Words ===")
preds = clf.predict(X_bow)
print(classification_report(labels, preds, target_names=['Negative', 'Positive']))

# --- What words are most predictive? ---
feature_names = bow.get_feature_names_out()
coefs = clf.coef_[0]
top_pos = sorted(zip(coefs, feature_names), reverse=True)[:5]
top_neg = sorted(zip(coefs, feature_names))[:5]

print("Most POSITIVE words:", [(w, f"{c:.2f}") for c, w in top_pos])
print("Most NEGATIVE words:", [(w, f"{c:.2f}") for c, w in top_neg])

---
# PART 6: Introduction to Transformers (Modern NLP)

Transformers are the architecture behind **ChatGPT, BERT, and Claude**.

The key innovation is the **Attention Mechanism**:
- Instead of reading words one-by-one, the model looks at **all words simultaneously**
- Each word figures out which other words are most relevant to it

Example for the sentence *"The bank can guarantee deposits will eventually cover future tuition costs"*:
- The word **"bank"** needs to decide: is this a riverbank or a financial institution?
- Attention lets it look at **"deposits"**, **"tuition"**, **"costs"** and figure out it's financial!

We'll use 🤗 **Hugging Face Transformers** — the standard library for pre-trained models.

In [ ]:
# ============================================================
# CELL 11: Using a pre-trained Transformer for sentiment
# ============================================================
# We'll use a small pre-trained model from Hugging Face.
# 'Pre-trained' means it was already trained on millions of texts.
# We just use it — no training required!
#
# If transformers is not installed:
# !pip install transformers --quiet

try:
    from transformers import pipeline

    # Load a pre-trained sentiment analysis pipeline
    # This downloads a small model the first time (~250MB)
    print("Loading pre-trained model... (first run downloads the model)")
    sentiment_pipeline = pipeline(
        "sentiment-analysis",
        model="distilbert-base-uncased-finetuned-sst-2-english"
    )

    test_sentences = [
        "I absolutely loved this course, it was brilliant!",
        "This lecture is boring and confusing.",
        "The content is okay, not great but not terrible either.",
        "Neural networks are fascinating and powerful!",
    ]

    print("\n=== Pre-trained Transformer Results ===")
    results = sentiment_pipeline(test_sentences)
    for sentence, result in zip(test_sentences, results):
        emoji = "😊" if result['label'] == 'POSITIVE' else "😞"
        print(f"{emoji} [{result['label']:8} {result['score']*100:5.1f}%] {sentence}")

except ImportError:
    print("transformers library not installed.")
    print("Run: pip install transformers")
    print("\nHere's what the output would look like:")
    print("😊 [POSITIVE  99.8%] I absolutely loved this course, it was brilliant!")
    print("😞 [NEGATIVE  97.2%] This lecture is boring and confusing.")

In [ ]:
# ============================================================
# CELL 12: Visualizing Attention (simplified)
# ============================================================
# Attention scores tell us: when processing word X,
# how much does the model 'look at' each other word?
# A score of 1.0 = full attention, 0.0 = ignore

import numpy as np
import matplotlib.pyplot as plt

# Simulated attention scores for the sentence:
# "The cat sat on the mat"
sentence = ["The", "cat", "sat", "on", "the", "mat"]

# Attention matrix: row = query word, col = key word
# (These are illustrative, not from a real model run)
attention = np.array([
    [0.4, 0.1, 0.1, 0.1, 0.2, 0.1],  # "The" looks mostly at itself
    [0.1, 0.5, 0.2, 0.1, 0.0, 0.1],  # "cat" looks at "sat" (what did cat do?)
    [0.1, 0.3, 0.3, 0.1, 0.0, 0.2],  # "sat" looks at "cat" and "mat"
    [0.1, 0.1, 0.1, 0.3, 0.1, 0.3],  # "on" looks at "mat"
    [0.3, 0.0, 0.0, 0.1, 0.4, 0.2],  # "the" looks at "mat" (which 'the'?)
    [0.1, 0.2, 0.2, 0.2, 0.1, 0.2],  # "mat" spread attention
])

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(attention, cmap='Blues', vmin=0, vmax=0.5)

ax.set_xticks(range(len(sentence)))
ax.set_yticks(range(len(sentence)))
ax.set_xticklabels(sentence, fontsize=12)
ax.set_yticklabels(sentence, fontsize=12)
ax.set_xlabel('Keys (words being attended to)', fontsize=11)
ax.set_ylabel('Queries (words doing the attending)', fontsize=11)
ax.set_title('Attention Map: "The cat sat on the mat"\n'
             'Darker = stronger attention', fontsize=12)

# Add text annotations
for i in range(len(sentence)):
    for j in range(len(sentence)):
        ax.text(j, i, f'{attention[i,j]:.1f}',
                ha='center', va='center', fontsize=9,
                color='white' if attention[i,j] > 0.35 else 'black')

plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

print("Key insight: 'cat' pays strong attention to 'sat' (the action the cat performed)")
print("This is how transformers understand relationships between words!")

---
# 🎓 Summary & What You've Learned

Congratulations! Here's everything covered in this notebook:

| Topic | Key Concept | Where you saw it |
|-------|-------------|------------------|
| Neuron | Weighted sum + bias | Cell 1 |
| Activation Functions | ReLU, Sigmoid | Cell 2 |
| Neural Network | Forward/backward pass | Cell 3 |
| PyTorch | `nn.Module`, optimizers | Cells 4–6 |
| NLP Preprocessing | Tokenize, encode, vocab | Cell 7 |
| Word Embeddings | word → vector | Cell 8 |
| Sentiment Analysis | End-to-end NLP model | Cell 9 |
| Bag of Words + TF-IDF | Classic NLP features | Cell 10 |
| Transformers | Pre-trained models | Cell 11 |
| Attention | How transformers 'read' | Cell 12 |

## 🚀 Next Steps
1. **Practice**: Modify the models — change hidden layer sizes, learning rates, epochs
2. **Bigger datasets**: Try the IMDB movie reviews dataset (25,000 reviews!)
3. **Fine-tuning**: Fine-tune a BERT model on your own data using Hugging Face
4. **Explore**: Try text generation, named entity recognition, question answering

## 📚 Recommended Resources
- [fast.ai](https://www.fast.ai) — Practical deep learning for free
- [Hugging Face Course](https://huggingface.co/learn) — NLP from basics to transformers
- [PyTorch Tutorials](https://pytorch.org/tutorials) — Official docs with examples
- *"Dive into Deep Learning"* — Free textbook at d2l.ai

In [ ]:
# ============================================================
# CELL 13: Practice Exercises — Try these yourself!
# ============================================================

print("="*60)
print("EXERCISES — modify this cell to complete them!")
print("="*60)

# EXERCISE 1: Single Neuron
# Change the weights and bias below. When does the umbrella decision flip?
ex1_inputs  = [1, 1]            # cloudy=1, summer=1
ex1_weights = [0.8, -0.3]       # Try: [0.5, 0.5]
ex1_bias    = -0.5              # Try: 0.0
ex1_output = sum(i*w for i, w in zip(ex1_inputs, ex1_weights)) + ex1_bias
print(f"\nEx1 - Neuron output: {ex1_output:.2f} → {'Umbrella 🌂' if ex1_output > 0 else 'No umbrella ☀️'}")

# EXERCISE 2: Add a new review to the sentiment dataset
# Add your own sentence and see how the model predicts it
my_review = "This was an interesting and thought provoking class"  # ← change this!
# (After running Cell 9, uncomment and run:)
# model_nlp.eval()
# tokens = tokenize(my_review)
# encoded = torch.LongTensor([encode(tokens, vocab, MAX_LEN)])
# with torch.no_grad():
#     score = model_nlp(encoded).item()
# print(f"\nEx2 - '{my_review}'")
# print(f"      Score: {score:.3f} → {'POSITIVE 😊' if score > 0.5 else 'NEGATIVE 😞'}")

# EXERCISE 3: Experiment with network size
# In Cell 6 (Iris), change the hidden layer sizes:
#   self.layer1 = nn.Linear(4, 32)   # bigger!
#   self.layer2 = nn.Linear(32, 16)  # bigger!
# Does accuracy improve? Does training take longer?
print("\nEx3 - Go to Cell 6 and change the hidden layer sizes. Re-run and compare!")

print("\nHappy experimenting! 🎉")